In [6]:
import tkinter as tk
from tkinter import messagebox
import sqlite3
import random

# DATABASE SETUP
def init_db():
    conn = sqlite3.connect("study_cards.db")
    cursor = conn.cursor()
    # Create table for storing terms and definitions
    cursor.execute("CREATE TABLE IF NOT EXISTS cards (term TEXT, definition TEXT)")
    conn.commit()
    conn.close()

# CUSTOM FUNCTIONS

def save_to_db():
    term = term_entry.get()
    definition = def_entry.get()

    if term == "" or definition == "":
        messagebox.showwarning("Input Error", "Please fill out both fields!")
        return

    conn = sqlite3.connect("study_cards.db")
    cursor = conn.cursor()
    cursor.execute("INSERT INTO cards (term, definition) VALUES (?, ?)", (term, definition))
    conn.commit()
    conn.close()

    # Clear inputs and update the count display
    term_entry.delete(0, tk.END)
    def_entry.delete(0, tk.END)
    update_count_label() 
    messagebox.showinfo("Success", f"Flashcard for '{term}' saved!")

def review_cards():
    conn = sqlite3.connect("study_cards.db")
    cursor = conn.cursor()
    cursor.execute("SELECT term, definition FROM cards")
    all_cards = cursor.fetchall()
    conn.close()

    if not all_cards:
        messagebox.showwarning("Empty", "No cards found! Add some first.")
        return

    
    random_card = random.choice(all_cards)
    
    
    if messagebox.askyesno("Review", f"Term: {random_card[0]}\n\nDo you want to see the definition?"):
        messagebox.showinfo("Definition", f"Definition: {random_card[1]}")

def update_count_label():
    """Function to check the database and update the UI label"""
    conn = sqlite3.connect("study_cards.db")
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM cards")
    count = cursor.fetchone()[0]
    conn.close()
    count_label.config(text=f"Total Cards in Database: {count}")

# APP UI
init_db() 
root = tk.Tk()
root.title("Cybersecurity Flashcard App")
root.geometry("400x450")

# Title
tk.Label(root, text="Study Flashcard Tool", font=("Arial", 16, "bold"), fg="#333").pack(pady=10)

# Database Check 
count_label = tk.Label(root, text="Total Cards in Database: 0", font=("Arial", 10, "italic"))
count_label.pack()
update_count_label() # Initial check when app opens

# Input Section
tk.Label(root, text="Enter Term:").pack(pady=(10, 0))
term_entry = tk.Entry(root, width=35)
term_entry.pack(pady=5)

tk.Label(root, text="Enter Definition:").pack()
def_entry = tk.Entry(root, width=35)
def_entry.pack(pady=5)

# Buttons
save_btn = tk.Button(root, text="Save to Database", command=save_to_db, bg="#4CAF50", fg="white", width=25)
save_btn.pack(pady=10)

review_btn = tk.Button(root, text="Review Random Card", command=review_cards, bg="#2196F3", fg="white", width=25)
review_btn.pack(pady=5)

# Exit
tk.Button(root, text="Exit", command=root.destroy, relief="flat").pack(pady=20)

root.mainloop()